# Sunspear RocketPy: Fixed airbrakes reference simulation

Simulation notebook from the Sunspear airbrakes control and estimation project.

In [ ]:
import rocketpy as rp
import datetime as dt
from rocketpy.utilities import apogee_by_mass
from rocketpy.simulation import FlightDataExporter

from pathlib import Path

# Works when Jupyter starts either in this folder or at the repository root.
_CWD = Path.cwd().resolve()
if (_CWD / "assets").is_dir():
    SIM_DIR = _CWD
elif (_CWD / "simulation" / "rocketpy" / "assets").is_dir():
    SIM_DIR = _CWD / "simulation" / "rocketpy"
else:
    raise FileNotFoundError("Start Jupyter from the repository root or simulation/rocketpy folder.")
REPO_ROOT = SIM_DIR.parents[1]
ASSET_DIR = SIM_DIR / "assets"
OUTPUT_DIR = SIM_DIR / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


airbrakes_triggered = False

## The Launch Environment

In [ ]:
domain = rp.Environment()
domain.set_location(39.3897,8.288964) # Coordinates of the EUROC Launch Site
#domain.set_elevation("Open-Elevation")
domain.set_elevation(elevation = 160.0)
today = dt.datetime.now()
domain.set_date(today, "CET")
#domain.set_atmospheric_model("Windy", "ICONEU")
print(domain.wind_speed)
domain.info()

## The Motor

In [ ]:
N2540 = rp.SolidMotor(
        thrust_source = str(ASSET_DIR / "Cesaroni_17907N2540-P.eng"),
        dry_mass = 5.581, #Rocket Mass without propellant (in kg)
        dry_inertia = (15.4789, 15.4789, 0.0294), #Rocket Inertia without propellant (in kg*m^2)
        nozzle_radius=75 / 2000,
        grain_number=6,
        grain_density= 2360,
        grain_outer_radius= 76 / 2000,
        grain_initial_inner_radius= 33 / 2000,
        grain_initial_height= 200 / 1000,
        grain_separation= 5 / 1000,
        grains_center_of_mass_position=0,
        center_of_dry_mass_position=0,
        nozzle_position=-0.62 - (4.44 / 100),
        burn_time= 7.222,
        throat_radius=20 / 2000,
        coordinate_system_orientation="nozzle_to_combustion_chamber",
)
N2540.info()

## The Rocket

In [ ]:
sunspear = rp.Rocket(
        radius = 11 / 200, #Radius (in m)
        mass = 9.35264, # Mass without motor (in kg)
        inertia = (13.342, 23.236, 15.523), #Inertia without motor (in kg*m^2)
        power_off_drag = str(ASSET_DIR / "cd_off_sunspear_v2.3a.CSV"),
        power_on_drag = str(ASSET_DIR / "cd_off_sunspear_v2.3a.CSV"),
        coordinate_system_orientation = "nose_to_tail",
        center_of_mass_without_motor = 1.23, #Center of Mass without Motor (in m)
    )
sunspear.add_motor(N2540, position = 2.75 - 0.62 - (4.44 / 100))
nose = sunspear.add_nose(
    length = 50 / 100,
    kind = 'von karman',
    position = 0
    )
fins = sunspear.add_trapezoidal_fins(
    n = 3,
    root_chord = 0.3,
    tip_chord = 0.1,
    span = 0.11,
    sweep_length = 0.173,
    position = 2.4,
    #cant_angle = 1
)
tail = sunspear.add_tail(
    top_radius = 10.2 / 200,
    bottom_radius = 7.54 / 200,
    length = 4.44 / 100,
    position = 2.75 - (4.44 / 100)
)
main = sunspear.add_parachute(
    name = "main",
    cd_s = 2.513,
    trigger = 450,
)
drogue = sunspear.add_parachute(
    name = "drogue",
    cd_s = 0.882,
    trigger = "apogee"
)
sunspear.info()
sunspear.draw()
sunspear.plots.stability_margin()

## Airbrakes control, constant input of 0.5

In [ ]:
BURN_TIME = 7.222  # s

def speed_of_sound_isa(altitude_m):
    gamma = 1.4
    R = 287.05
    T0 = 288.15
    lapse = -0.0065

    h = max(0.0, altitude_m)
    T = T0 + lapse * h
    T = max(T, 216.65)
    return (gamma * R * T) ** 0.5

def controller_50pct_coast_mach(time, sampling_rate, state, state_history, observed_variables, air_brakes):
    global airbrakes_triggered

    altitude = float(state[2])
    vx = float(state[3])
    vy = float(state[4])
    vz = float(state[5])

    speed = (vx**2 + vy**2 + vz**2) ** 0.5
    mach = speed / speed_of_sound_isa(altitude)

    condition = (time >= BURN_TIME and mach < 0.95 and vz > 0) 

    if condition:
        air_brakes.deployment_level = 0.8

        # 🔥 PRINT SOLO UNA VEZ
        if not airbrakes_triggered:
            print(f"[TRIGGER] t={time:.3f}s | Mach={mach:.3f} | z={altitude:.2f} | vz={vz:.2f}")
            airbrakes_triggered = True

    else:
        air_brakes.deployment_level = 0.0


    return observed_variables

## Airbrakes aerodynamic data

In [ ]:
A_MAX = 1e-1  # m²

air_brakes = sunspear.add_air_brakes(
drag_coefficient_curve=[
    # Mach = 0.00
    [0.0, 0.00, 0.000],
    [0.1, 0.00, 0.025],
    [0.2, 0.00, 0.050],
    [0.3, 0.00, 0.075],
    [0.4, 0.00, 0.100],
    [0.5, 0.00, 0.125],
    [0.6, 0.00, 0.150],
    [0.7, 0.00, 0.175],
    [0.8, 0.00, 0.200],
    [0.9, 0.00, 0.225],
    [1.0, 0.00, 0.250],

    # Mach = 0.94
    [0.0, 0.94, 0.000],
    [0.1, 0.94, 0.025],
    [0.2, 0.94, 0.050],
    [0.3, 0.94, 0.075],
    [0.4, 0.94, 0.100],
    [0.5, 0.94, 0.125],
    [0.6, 0.94, 0.150],
    [0.7, 0.94, 0.175],
    [0.8, 0.94, 0.200],
    [0.9, 0.94, 0.225],
    [1.0, 0.94, 0.250],

    # Mach = 0.95
    [0.0, 0.95, 0.000],
    [0.1, 0.95, 0.000],
    [0.2, 0.95, 0.000],
    [0.3, 0.95, 0.000],
    [0.4, 0.95, 0.000],
    [0.5, 0.95, 0.000],
    [0.6, 0.95, 0.000],
    [0.7, 0.95, 0.000],
    [0.8, 0.95, 0.000],
    [0.9, 0.95, 0.000],
    [1.0, 0.95, 0.000],

    # Mach = 1.20
    [0.0, 1.20, 0.000],
    [0.1, 1.20, 0.000],
    [0.2, 1.20, 0.000],
    [0.3, 1.20, 0.000],
    [0.4, 1.20, 0.000],
    [0.5, 1.20, 0.000],
    [0.6, 1.20, 0.000],
    [0.7, 1.20, 0.000],
    [0.8, 1.20, 0.000],
    [0.9, 1.20, 0.000],
    [1.0, 1.20, 0.000],
],
    controller_function=controller_50pct_coast_mach,
    sampling_rate=20,
    reference_area=A_MAX,
    clamp=True,
    override_rocket_drag=False,
    initial_observed_variables=(0.0, 0.0),
    name="Airbrakes 10%-step (Coast, Mach<0.95)",
)


## The Simulation

In [ ]:
launch = rp.Flight(rocket = sunspear, environment = domain, rail_length = 12, inclination = 84, heading = 133)
launch.prints.initial_conditions()
launch.prints.surface_wind_conditions()
launch.prints.launch_rail_conditions()

**Launch Rail Conditions**

In [ ]:
launch.prints.out_of_rail_conditions()

**Apogee**

In [ ]:
launch.prints.apogee_conditions()
launch.plots.trajectory_3d()

In [ ]:
launch.prints.maximum_values()

**Parachutes**

In [ ]:
launch.prints.events_registered()

**Touchdown**

In [ ]:
launch.prints.impact_conditions()

In [ ]:
exporter = FlightDataExporter(launch)


exporter.export_data(
    str(OUTPUT_DIR / "reference_flight.csv"),
    "x",
    "y",
    "z",
    "e0",
    "e1",
    "e2",
    "e3",
    "w1",
    "w2",
    "w3",
    time_step=0.01
)

## Plots

In [ ]:
launch.plots.stability_and_control_data()

In [ ]:
launch.plots.aerodynamic_forces()

In [ ]:
launch.plots.linear_kinematics_data()

In [ ]:
launch.plots.flight_path_angle_data()

## Google Earth File

In [ ]:
KML_DIR = OUTPUT_DIR / "trajectory"
KML_DIR.mkdir(parents=True, exist_ok=True)
KML_DIR = OUTPUT_DIR / "trajectory"
KML_DIR.mkdir(parents=True, exist_ok=True)
launch.export_kml(
    file_name = str(KML_DIR / "trajectory.kml"),
    extrude = True,
    altitude_mode = "relative_to_ground"
)

## Apogee vs Mass

In [ ]:
res = apogee_by_mass(flight = launch, min_mass = 11, max_mass = 16, points = 20, plot = True)
m = 10
while m < 18:
    apg = res.get_value(m)
    if apg < 9100:
        print(f"Maximum Mass = {m}kg")
        break
    m += 0.01